# AmSC Data Catalog Tutorial

This notebook demonstrates how to use the AmSC Python Client to manage
scientific data in the AmSC Data Catalog, which is powered by
[OpenMetadata](https://open-metadata.org/).

**What you'll do:**
1. Create an AmSC client with Globus authentication
2. Create a ScientificWork (top-level container for a research project)
3. Create Artifacts (individual datasets) under the ScientificWork
4. Search the catalog
5. Retrieve entity details
6. Delete entities

**Prerequisites:**
- Install dependencies: `pip install -r requirements.txt`
- A [Globus](https://www.globus.org/) account
- Write access to a data catalog

**Authentication:**
A browser window will open for Globus login on first API call.
Tokens are cached locally for subsequent runs.

**Catalog Hierarchy:**
```
Catalog (e.g., olcf-constellation-amsc-storage.olcf-const-data-catalog)
  └─ ScientificWork (research project / study)
       ├─ Artifact (individual dataset file)
       ├─ Artifact
       └─ ArtifactCollection (group of related artifacts)
```

In [ ]:
from amsc_client import Client

## Step 1: Create the AmSC Client

The client uses Globus OAuth2 for authentication. The required environment
variables specify which Globus application and scope to use.

On first API call, a browser window will open for Globus login.

In [ ]:
# Step 1: Create client (triggers Globus login if needed)
print("\nStep 1: Creating AmSC client...")

base_url = 'https://api.american-science-cloud.org/api/current'
globus_app_id = 'e4f48665-38b5-4833-a89e-849c71f5b3e3'  # AmSC CLI Client

client = Client(
    base_url=base_url,
    auth_method="globus",
    globus_client_id=globus_app_id,
    requested_scopes=f'openid profile email https://auth.globus.org/scopes/{globus_app_id}/amsc_test',
    resource_server=globus_app_id,
    use_id_token=True,
)
print(f"  ✅ Connected to {base_url}")

## Step 2: Create a ScientificWork

A **ScientificWork** is the top-level container in the catalog hierarchy.
It represents a research project, study, or experiment. Artifacts
(individual datasets) are created as children underneath it.

Each entity has a **Fully Qualified Name (FQN)** that uniquely identifies
it within the catalog, e.g.:
```
olcf-constellation-amsc-storage.olcf-const-data-catalog.test-soil-analysis-20260328
```

In [ ]:
CATALOG = "olcf-constellation-amsc-storage.olcf-const-data-catalog"

# Unique suffix for each run so repeated invocations don't collide
import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# Step 2: Create a ScientificWork
work_name = f"test-soil-analysis-{RUN_ID}"
print(f"\nStep 2: Creating ScientificWork '{work_name}'...")
try:
    result = client.catalog.create_scientific_work(
        catalog=CATALOG,
        name=work_name,
        description="Test scientific work for soil composition analysis across multiple sites",
        location=f"s3://test-bucket/soil-analysis-{RUN_ID}/",
        display_name=f"Test Soil Analysis ({RUN_ID})",
    )
    print(f"  ✅ Created ScientificWork")
    print(f"     FQN: {result.fqn}")
    print(f"     Message: {result.message}")
except Exception as e:
    print(f"  ❌ Failed: {e}")
        

## Step 3: Create Artifacts

**Artifacts** represent individual datasets (files, tables, etc.) within a
ScientificWork. Each artifact has:

| Field | Description |
|-------|-------------|
| `name` | Unique identifier within the parent |
| `description` | Human-readable description |
| `location` | Where the data lives (S3 URL, Globus URL, etc.) |
| `format` | MIME type (e.g., `application/parquet`, `text/csv`) |
| `size` | Size in bytes |
| `parent` | FQN of the parent ScientificWork |

Here we create three artifacts representing different soil analysis datasets.

In [ ]:
# Step 3: Create 3 Artifacts under the ScientificWork
artifact_defs = [
    {
        "name": f"test-soil-moisture-readings-{RUN_ID}",
        "description": "Soil moisture sensor readings from 12 field stations, hourly intervals",
        "display_name": f"Soil Moisture Readings ({RUN_ID})",
        "location": f"s3://test-bucket/soil-analysis-{RUN_ID}/moisture-readings.parquet",
        "format": "application/parquet",
        "size": 2097152,
    },
    {
        "name": f"test-soil-nutrient-profiles-{RUN_ID}",
        "description": "NPK nutrient concentration profiles from soil core samples",
        "display_name": f"Soil Nutrient Profiles ({RUN_ID})",
        "location": f"s3://test-bucket/soil-analysis-{RUN_ID}/nutrient-profiles.csv",
        "format": "text/csv",
        "size": 524288,
    },
    {
        "name": f"test-soil-ph-measurements-{RUN_ID}",
        "description": "Soil pH measurements across agricultural test plots, seasonal data",
        "display_name": f"Soil pH Measurements ({RUN_ID})",
        "location": f"s3://test-bucket/soil-analysis-{RUN_ID}/ph-measurements.json",
        "format": "application/json",
        "size": 131072,
    },
]

print(f"\nStep 3: Creating {len(artifact_defs)} Artifacts...")
artifact_fqns = []
cc_result = result
for i, art in enumerate(artifact_defs, 1):
    try:
        result = client.catalog.create_artifact(
            catalog=CATALOG,
            parent=cc_result.fqn,
            **art,
        )
        artifact_fqns.append(result.fqn)
        print(f"  ✅ [{i}/{len(artifact_defs)}] Created '{art['name']}'")
        print(f"     FQN: {result.fqn}")
    except Exception as e:
        print(f"  ❌ [{i}/{len(artifact_defs)}] Failed '{art['name']}': {e}")


## Step 4: Search the Catalog

The `search()` method returns a `SearchResults` object that supports:
- **Iteration**: `for item in results`
- **Length**: `len(results)` (items in this page)
- **Indexing**: `results[0]`
- **Total count**: `results.total_count` (total matches across all pages)

Each result has user-friendly property aliases:
- `item.type` (instead of the autogen's `item.type_`)
- `item.format` (instead of `item.format_`)

In [ ]:
# Step 4: Search for artifacts
print(f"\nStep 4: Searching for 'soil'...")
try:
    results = client.catalog.search("soil", limit=10)
    print(f"  Total results: {results.total_count}")
    print(f"  This page: {len(results)}")
    for item in results:
        print(f"    - {item.name} ({item.type})")
        print(f"      FQN: {item.fqn}")
except Exception as e:
    print(f"  ⚠️  Search failed: {e}")

## Step 5: Retrieve Entity Details

Use `client.catalog.get(fqn)` to fetch the full details of any entity.
The returned object is a typed wrapper (`Artifact`, `ScientificWork`, etc.)
with all the metadata fields.

In [ ]:
# Step 5: Get details for each artifact
print(f"\nStep 5: Getting details for each artifact...")
for fqn in artifact_fqns:
    try:
        entity = client.catalog.get(fqn)
        print(f"  ✅ {entity.name}")
        print(f"     Description: {entity.description}")
        print(f"     Location:    {entity.location}")
        print(f"     Format:      {entity.format}")
        print(f"     Size:        {getattr(entity, 'size', 'N/A')} bytes")
        print(f"     FQN:         {entity.fqn}")
    except Exception as e:
        print(f"  ❌ Failed to get {fqn}: {e}")

## Step 6: Delete an Entity

Use `client.catalog.delete(fqn)` to remove an entity from the catalog.

**Note:** Deleting a ScientificWork also deletes all of its children
(artifacts and collections). Here we delete only the first artifact
to demonstrate the operation, and leave the rest for inspection.

In [ ]:
# Step 6: Delete only the FIRST artifact
print(f"\nStep 6: Deleting first artifact only...")
if artifact_fqns:
    fqn_to_delete = artifact_fqns[0]
    try:
        result = client.catalog.delete(fqn_to_delete)
        print(f"  ✅ Deleted: {fqn_to_delete}")
        print(f"     Message: {result.message}")
    except Exception as e:
        print(f"  ⚠️  Failed to delete: {e}")

    print(f"\n  Remaining entities (not deleted):")
    print(f"    ScientificWork: {cc_result.fqn}")
    for fqn in artifact_fqns[1:]:
        print(f"    Artifact:       {fqn}")


## Summary

This tutorial showed how to perform full CRUD operations on the AmSC Data Catalog:

| Operation | API Call | Returns |
|-----------|---------|----------|
| Create ScientificWork | `client.catalog.create_scientific_work(...)` | `CatalogResponse` with `.fqn` |
| Create Artifact | `client.catalog.create_artifact(...)` | `CatalogResponse` with `.fqn` |
| Search | `client.catalog.search(query)` | `SearchResults` (iterable) |
| Get details | `client.catalog.get(fqn)` | `Artifact`, `ScientificWork`, etc. |
| Delete | `client.catalog.delete(fqn)` | `CatalogResponse` |

**Key concepts:**
- All entities live within a **catalog** (e.g., `olcf-constellation-amsc-storage.olcf-const-data-catalog`)
- **ScientificWorks** are top-level containers; **Artifacts** are children within them
- Every entity is identified by its **FQN** (fully qualified name)
- `SearchResults` supports iteration, `len()`, indexing, and `.total_count`
- Property aliases (`item.type`, `item.format`) make the API Pythonic — no need to use `type_` or `format_`
- Everything is imported from `amsc_client` — no need to import from `amsc_api_autogen`